In [2]:
import pandas as pd 

df = pd.read_csv("jpboxcleanmaxV2.csv")


In [3]:
# Compter le nombre de lignes où "Démarrage" est manquant
missing_demarrage_count = df["Démarrage"].isnull().sum()

print(f"Nombre de lignes où 'Démarrage' est manquant : {missing_demarrage_count}")


Nombre de lignes où 'Démarrage' est manquant : 94


In [4]:
# Supprimer les lignes où "Démarrage" est manquant
df = df.dropna(subset=["Démarrage"])


In [5]:
# Compter le nombre de lignes où "Démarrage" est manquant
missing_demarrage_count = df["Démarrage"].isnull().sum()

print(f"Nombre de lignes où 'Démarrage' est manquant : {missing_demarrage_count}")


Nombre de lignes où 'Démarrage' est manquant : 0


In [6]:
df.to_parquet("jpboxmax.parquet", engine="pyarrow", index=False)

In [7]:
# Charger le fichier Parquet
df_parquet = pd.read_parquet("jpboxmax.parquet")
print(df_parquet.head())


                         Titre  Démarrage  Entrées totales  \
0                     scream 4   533026.0          1070890   
1                resident evil   533300.0          1105943   
2  gainsbourg - (vie héroïque)   511713.0          1199451   
3         matrix resurrections   513133.0           972644   
4           casse-tête chinois   513407.0          1536489   

            Réalisateur        Pays               Genre Date de sortie  \
0            Wes Craven  Etats-Unis             Horreur     2011-04-13   
1    Paul W.S. Anderson  Etats-Unis   Aventure - Action     2002-04-03   
2                  None      France  Comédie dramatique     2010-01-20   
3  Wachowski (brothers)  Etats-Unis     Science Fiction     2021-12-22   
4       Cédric Klapisch      France             Comédie     2013-12-04   

    Classification  Distributeur  Rang toutes exploitations  Rang général  \
0  Moins de 12 ans           SND                        NaN           NaN   
1  Moins de 12 ans  Metropol

In [8]:
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


In [9]:
# Charger le fichier parquet
df = pd.read_parquet("jpboxmax.parquet")


In [10]:
print(type(df))
print(df.head())  # Affiche les 5 premières lignes si `df` est valide


<class 'pandas.core.frame.DataFrame'>
                         Titre  Démarrage  Entrées totales  \
0                     scream 4   533026.0          1070890   
1                resident evil   533300.0          1105943   
2  gainsbourg - (vie héroïque)   511713.0          1199451   
3         matrix resurrections   513133.0           972644   
4           casse-tête chinois   513407.0          1536489   

            Réalisateur        Pays               Genre Date de sortie  \
0            Wes Craven  Etats-Unis             Horreur     2011-04-13   
1    Paul W.S. Anderson  Etats-Unis   Aventure - Action     2002-04-03   
2                  None      France  Comédie dramatique     2010-01-20   
3  Wachowski (brothers)  Etats-Unis     Science Fiction     2021-12-22   
4       Cédric Klapisch      France             Comédie     2013-12-04   

    Classification  Distributeur  Rang toutes exploitations  Rang général  \
0  Moins de 12 ans           SND                        NaN        

In [11]:
# Définir la liste des colonnes à ignorer
cols_to_drop = [
    "Entrées totales", 
    "Rang toutes exploitations", 
    "Rang général", 
    "Nb 1ère places hebdo", 
    "Nb semaines Top20", 
    "Meilleure place hebdo"
]

# Supprimer ces colonnes du DataFrame
df.drop(columns=cols_to_drop, inplace=True)


In [12]:
# Convertir la colonne "Date de sortie" en datetime
df["Date de sortie"] = pd.to_datetime(df["Date de sortie"], errors="coerce")

# Extraire des informations utiles de la date
df["Year"] = df["Date de sortie"].dt.year
df["Month"] = df["Date de sortie"].dt.month
df["Day"] = df["Date de sortie"].dt.day
df["DayOfWeek"] = df["Date de sortie"].dt.dayofweek  # Lundi=0, Dimanche=6

# Vous pouvez choisir de conserver ou de supprimer la colonne original si vous avez extrait toutes les infos
# df.drop(columns=["Date de sortie"], inplace=True)


In [13]:
target = "Démarrage"
# Sélectionner toutes les colonnes sauf la cible
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]


In [14]:
categorical_features = [
    "Titre", "Réalisateur", "Pays", "Genre", 
    "Classification", "Distributeur", "Acteur principal", "Acteur secondaire"
]


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [17]:
# Remplir les valeurs catégoriques manquantes avec "Unknown"
for col in X_train.select_dtypes(include=["object"]).columns:
    X_train[col].fillna("Unknown", inplace=True)
    X_test[col].fillna("Unknown", inplace=True)


/tmp/ipykernel_157016/3682422638.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train[col].fillna("Unknown", inplace=True)
/tmp/ipykernel_157016/3682422638.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [18]:
# Création du Pool d'entraînement et de test
train_pool = Pool(data=X_train, label=y_train, cat_features=categorical_features)
test_pool = Pool(data=X_test, label=y_test, cat_features=categorical_features)


In [20]:
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=4,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)

# Entraîner le modèle avec le Pool d'entraînement et évaluer sur le Pool de test
model.fit(train_pool, eval_set=test_pool, use_best_model=True)


0:	learn: 288950.6759025	test: 319860.9252142	best: 319860.9252142 (0)	total: 6.56ms	remaining: 6.55s
100:	learn: 225135.4632159	test: 255246.4528307	best: 255246.4528307 (100)	total: 296ms	remaining: 2.63s
200:	learn: 215693.4395894	test: 250331.7862509	best: 250331.7862509 (200)	total: 620ms	remaining: 2.46s
300:	learn: 211849.2138877	test: 249517.1924150	best: 249509.7585345 (298)	total: 942ms	remaining: 2.19s
400:	learn: 206421.2985143	test: 246568.3813676	best: 246528.8775255 (398)	total: 1.31s	remaining: 1.96s
500:	learn: 202026.3167751	test: 245457.7124692	best: 245452.9768312 (496)	total: 1.72s	remaining: 1.72s
600:	learn: 198112.5467074	test: 244268.7753867	best: 244259.3193260 (596)	total: 2.11s	remaining: 1.4s
700:	learn: 194591.6608068	test: 243301.3478835	best: 243301.3478835 (700)	total: 2.57s	remaining: 1.1s
800:	learn: 191369.0277785	test: 242644.6503948	best: 242629.7862858 (799)	total: 2.94s	remaining: 729ms
900:	learn: 188956.0849794	test: 242200.6914616	best: 242190